# Classify using DTW on stretched spike first deriv.

In [ ]:
cut_params = {
    'signal_name': 'wideband',
    'event_name': 'spk',
    'cut_window': [-0.001, 0.002],
    'fs': 40000
}
proc_params = {
    'smooth_params': {
        'window_length': 9,
        'polyorder': 3
    },
    'dv_threshold': 0.1
}

In [ ]:
import os

INVITRO_DATA_DIR = f'{os.environ["RAAP_INVITRO_DATA"]}/juxta'
INVIVO_DATA_DIR = f'{os.environ["RAAP_INVIVO_DATA"]}'
INVIVO_FEATURE_DIR = f'{os.environ["RAAP_INVIVO_FEATURES"]}'

In [ ]:
training_cell_names = [
    '2406241PZ_01',
    '2406242PZ_03',
    '2406263PZ_01',
    '2406282PZ_01',
    '2407031PZ_01',
    '2407031SI_gr2',
    '2407041SI_gr1'
]

In [ ]:
from workflow import load_training_dataset
training_dataset = load_training_dataset(
    training_cell_names,
    INVITRO_DATA_DIR,
    cut_params=cut_params,
    proc_params=proc_params
)

In [ ]:
from transformers import (
    FirstDeriv,
    rising_phase,
    first_derivative,
    resample_first_deriv,
    normalize_first_deriv
)
test_ais = training_dataset[0]['AISAP'][0]
test_ra = training_dataset[0]['RAAP'][0]
ais_deriv = first_derivative(rising_phase(test_ais, start=0.15, end=0.85))
ra_deriv = first_derivative(rising_phase(test_ra, start=0.15, end=0.85))
ais_deriv_norm = normalize_first_deriv(ais_deriv)
ra_deriv_norm = normalize_first_deriv(ra_deriv)
ais_deriv_resampled = resample_first_deriv(ais_deriv_norm, 20)
ra_deriv_resampled = resample_first_deriv(ra_deriv_norm, 20)

In [ ]:
from dtw import dtw
raw_alignment = dtw(
    ais_deriv.dv,
    ra_deriv.dv,
    dist_method='euclidean'
)
norm_alignment = dtw(
    ais_deriv_norm.dv,
    ra_deriv_norm.dv,
    dist_method='euclidean'
)
resampled_alignment = dtw(
    ais_deriv_resampled.dv,
    ra_deriv_resampled.dv,
    dist_method='euclidean'
)
unnorm_dist = getattr(raw_alignment, 'distance')
norm_dist = getattr(norm_alignment, 'normalizedDistance')
resampled_dist = getattr(resampled_alignment, 'distance')
print(f'Unnormalized DTW distance: {unnorm_dist}')
print(f'Normalized DTW distance: {norm_dist}')
print(f'Resampled DTW distance: {resampled_dist}')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(6, 6))
axes[0].plot(
    ais_deriv.times,
    ais_deriv.dv,
    color='C0'
)
axes[0].plot(
    ra_deriv.times,
    ra_deriv.dv,
    color='C1'
)
axes[1].plot(
    ais_deriv_norm.times,
    ais_deriv_norm.dv,
    color='C0'
)
axes[1].plot(
    ais_deriv_resampled.times,
    ais_deriv_resampled.dv,
    '--',
    color='black'
)
axes[1].plot(
    ra_deriv_resampled.times,
    ra_deriv_resampled.dv,
    color='C1'
)
axes[1].plot(
    ra_deriv_resampled.times,
    ra_deriv_resampled.dv,
    '--',
    color='black'
)
plt.show()

In [ ]:
import numpy as np
n_training_samples = len(training_dataset)
derivs, y = [], []
for i, spike_type in enumerate(['AISAP', 'RAAP']):
    for cell_shapes in training_dataset:
        for spike in cell_shapes[spike_type]:
            deriv = resample_first_deriv(
                normalize_first_deriv(
                    first_derivative(
                        rising_phase(spike)
                    )
                ), 20
            )
            derivs.append(deriv)
            y.append(i)
y = np.array(y)
n_spikes = len(derivs)

In [ ]:
from tqdm import tqdm
D = np.zeros((n_spikes, n_spikes))
for i in tqdm(range(n_spikes), total=n_spikes):
    for j in range(i + 1, n_spikes):
        alignment = dtw(
            derivs[i].dv,
            derivs[j].dv,
            dist_method='euclidean'
        )
        D[i, j] = getattr(alignment, 'distance')
        D[j, i] = D[i, j]

In [ ]:
group_mean_curves = []
for yy in [0, 1]:
    group_curves = [curve for curve, label in zip(derivs, y) if label == yy]
    group_mean_curves.append(
        normalize_first_deriv(FirstDeriv(
            np.mean([curve.times for curve in group_curves], axis=0),
            np.mean([curve.dv for curve in group_curves], axis=0)
        ))
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 6))
for deriv, label in zip(derivs, y):
    axes[label].plot(
        group_mean_curves[label].times,
        group_mean_curves[label].dv,
        color='C0' if label == 0 else 'C1',
        alpha=0.3
    )
plt.show()

In [ ]:
import pandas as pd
D_from_mean = np.zeros((n_spikes, 2))
for i, deriv in tqdm(enumerate(derivs), total=n_spikes):
    for yy in [0, 1]:
        alignment = dtw(
            deriv.dv,
            group_mean_curves[yy].dv,
            dist_method='euclidean'
        )
        D_from_mean[i, yy] = getattr(alignment, 'distance')

mean_dist_df = pd.DataFrame(np.stack([D_from_mean[:, 0], D_from_mean[:, 1], y], axis=1), columns=['AISAP', 'RAAP', 'Label'])
mean_dist_df['id'] = mean_dist_df.index
mean_dist_df['pref'] = (
    mean_dist_df['AISAP'] - mean_dist_df['RAAP']
) / (
    mean_dist_df['AISAP'] + mean_dist_df['RAAP']
)

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(
    data=mean_dist_df,
    x='id',
    y='pref',
    hue='Label',
    ax=ax
)
plt.show()